# HW2 — Profile & Optimize an Autoregressive Decode Loop

**Lecture mapping:** L1 §07 (Profiling) · L2 §01 (prefill/decode, KV cache) · L2 §03 (engine optimizations)

The harness below defines a deliberately slow greedy-decode loop (**V0**): no KV
cache (it recomputes the whole sequence every step), fp32 eager, and a host sync
every step. Profile it, find the bottlenecks, and write a fast, numerically
identical replacement.

## What you implement

| Part | Function |
|------|----------|
| 1 | `profile` — wrap a loop in `torch.profiler`, print a table, export a Chrome trace |
| 2 | `optimized_loop` — fast greedy decode, same tokens as V0 (fp32) |
| 3 | `generate_optimized` — build, warm up, and time your optimized generation |
| 4 | Writeup Q1–Q4 |

## Speedup targets

`optimized_loop` end-to-end vs the V0 baseline:

| Speedup vs V0 | |
|---------------|--|
| ≥ 4.0× | excellent |
| ≥ 3.0× | good |
| ≥ 1.5× | some speedup |
| < 1.5× | little to no speedup |

The speedup only counts if `optimized_loop` passes the fp32 correctness check and
the timed run is on a GPU.

## Rules

- PyTorch + `requirements.txt` only — no vLLM / TensorRT-LLM / SGLang.
- Don't edit the harness cell.
- `optimized_loop` must reproduce the baseline's greedy tokens exactly on the same
  fp32 model. `generate_optimized` may switch to bf16 for the timed run.

## Where to look

- **KV cache** (L2 §01) — the baseline throws it away every step. Biggest win.
- **Host syncs** (L1 §07) — `.item()` on the critical path serializes CPU↔GPU.
- **`torch.compile` / CUDA graphs** (L2 §03) — fuse ops, cut launch overhead.
- **dtype** — bf16 for the timed run (small on a model this size).

## Cell types

- **DO NOT EDIT** — fixed harness.
- **YOUR IMPLEMENTATION** — replace `raise NotImplementedError`.
- **SELF-CHECK** — asserts that must pass.
- **WRITEUP** — answer in the markdown cell.

Run top-to-bottom on the GPU. Submit the executed notebook plus the files under
`results/`.

## The fixed harness (DO NOT EDIT)

Tiny 2-layer Llama, the V0 baseline, the correctness check, the timing helper,
and the speedup targets.

In [1]:
# DO NOT EDIT — editing this cell invalidates your speedup numbers.
import os, time

import torch
from transformers import LlamaConfig, LlamaForCausalLM

notebook_path = globals()['__vsc_ipynb_file__']
os.chdir(os.path.dirname(notebook_path))

RESULTS_DIR = os.path.join("results", "hw2")
os.makedirs(RESULTS_DIR, exist_ok=True)

SEED = 0
PROMPT_LEN = 64
MAX_NEW_TOKENS = 128

# Speedup targets (optimized end-to-end vs V0 baseline).
TIERS = [
    (4.0, ">= 4.0x  (excellent)"),
    (3.0, ">= 3.0x  (good)"),
    (1.5, ">= 1.5x  (some speedup)"),
    (0.0, "< 1.5x  (little to no speedup)"),
]


def device() -> str:
    return "cuda" if torch.cuda.is_available() else "cpu"


def _config() -> LlamaConfig:
    # Small enough to iterate fast, real enough to profile meaningfully.
    return LlamaConfig(
        vocab_size=32000,
        hidden_size=512,
        intermediate_size=1376,
        num_hidden_layers=2,
        num_attention_heads=8,
        num_key_value_heads=8,
        max_position_embeddings=4096,
    )


def build_model_and_input(dtype: torch.dtype = torch.float32):
    """Return (model, input_ids) on the active device with fixed random weights.

    The same SEED is used every call, so the V0 baseline and your optimized run
    see identical weights and the same prompt.
    """
    torch.manual_seed(SEED)
    model = LlamaForCausalLM(_config()).to(device=device(), dtype=dtype).eval()
    input_ids = torch.randint(0, 32000, (1, PROMPT_LEN), device=device())
    return model, input_ids


@torch.no_grad()
def baseline_loop(model, input_ids, max_new_tokens: int) -> torch.Tensor:
    """V0 — intentionally slow. Greedy decode with NO KV cache: every step
    re-runs the model over the whole growing sequence (O(n^2)), and forces a
    host sync each step (the `.item()`) — exactly the anti-patterns from lecture.

    Returns the generated token ids, shape (1, max_new_tokens).
    """
    generated = input_ids
    out_tokens = []
    for _ in range(max_new_tokens):
        out = model(input_ids=generated, use_cache=False)
        next_tok = out.logits[:, -1:, :].argmax(dim=-1)
        _ = next_tok.item()                       # forced host sync every step
        generated = torch.cat([generated, next_tok], dim=1)
        out_tokens.append(next_tok)
    return torch.cat(out_tokens, dim=1)


def time_loop(loop_fn, *args, n_warmup: int = 1, n_iters: int = 3) -> float:
    """Average seconds per full generation of `loop_fn(*args)`."""
    for _ in range(n_warmup):
        loop_fn(*args)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        for _ in range(n_iters):
            loop_fn(*args)
        end.record()
        torch.cuda.synchronize()
        return (start.elapsed_time(end) / 1e3) / n_iters
    t0 = time.perf_counter()
    for _ in range(n_iters):
        loop_fn(*args)
    return (time.perf_counter() - t0) / n_iters


def check_correctness(reference: torch.Tensor, candidate: torch.Tensor) -> bool:
    """Greedy decoding is deterministic, so a correct optimized loop must
    reproduce the baseline's token ids EXACTLY when given the same fp32 model."""
    return (reference.shape == candidate.shape
            and torch.equal(reference.cpu(), candidate.cpu()))


def speedup_tier(speedup: float) -> str:
    for threshold, label in TIERS:
        if speedup >= threshold:
            return label
    return TIERS[-1][1]


print(f"device={device()}  prompt_len={PROMPT_LEN}  new_tokens={MAX_NEW_TOKENS}")

/home/labs/antebilab/guyilan/Courses/Nebius/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device=cuda  prompt_len=64  new_tokens=128


## Part 1 — profile a generation loop

See the L1 §07 `torch.profiler` snippet. You will use this on both the baseline
and your optimized loop, and read the traces at
[ui.perfetto.dev](https://ui.perfetto.dev).

In [2]:
# TODO-CELL: profile
# YOUR IMPLEMENTATION
def profile(loop_fn, model, input_ids, max_new_tokens: int, trace_path: str) -> None:
    """Run `loop_fn(model, input_ids, max_new_tokens)` under torch.profiler.

    Requirements:
      * Profile both CPU and CUDA activity (ProfilerActivity).
      * Print a short table sorted by self CUDA time (key_averages().table(...))
        — on CPU-only machines sort by self CPU time instead.
      * Export a Chrome trace to `trace_path`.
    """
    activities = [torch.profiler.ProfilerActivity.CPU]
    use_cuda = torch.cuda.is_available()
    if use_cuda:
        activities.append(torch.profiler.ProfilerActivity.CUDA)

    with torch.profiler.profile(activities=activities, record_shapes=False) as prof:
        loop_fn(model, input_ids, max_new_tokens)
        if use_cuda:
            torch.cuda.synchronize()

    sort_key = "self_cuda_time_total" if use_cuda else "self_cpu_time_total"
    print(prof.key_averages().table(sort_by=sort_key, row_limit=15))

    prof.export_chrome_trace(trace_path)

## Part 2 — a fast, correct greedy decode loop

Return the same token ids as `baseline_loop` for the same fp32 model — shape
`(1, max_new_tokens)`. Start with what the baseline recomputes every step and
what it forces onto the host.

In [3]:
# TODO-CELL: optimized_loop
# YOUR IMPLEMENTATION
@torch.no_grad()
def optimized_loop(model, input_ids, max_new_tokens: int) -> torch.Tensor:
    """Greedy decode, but fast. Same tokens as baseline_loop, much less work."""
    out = model(input_ids=input_ids, use_cache=True)
    past_key_values = out.past_key_values
    next_tok = out.logits[:, -1:, :].argmax(dim=-1)

    out_tokens = [next_tok]
    cur_tok = next_tok

    for _ in range(max_new_tokens - 1):
        out = model(input_ids=cur_tok, past_key_values=past_key_values, use_cache=True)
        past_key_values = out.past_key_values
        cur_tok = out.logits[:, -1:, :].argmax(dim=-1)
        out_tokens.append(cur_tok)

    return torch.cat(out_tokens, dim=1)

## Part 3 — build + time the optimized generation

Build the model however you like (dtype, `torch.compile`, CUDA graphs), run
`optimized_loop`, and return `(elapsed_seconds, generated_token_ids)`.

`elapsed_seconds` must be a warmed-up, timed measurement (use `time_loop`); it's
what gets compared to V0. Numerics-changing tricks (e.g. bf16) are fine here —
correctness is checked separately on the fp32 path.

In [4]:
# TODO-CELL: generate_optimized
# YOUR IMPLEMENTATION
def generate_optimized(max_new_tokens: int = MAX_NEW_TOKENS):
    """Return (elapsed_seconds, generated_token_ids) for your fastest setup."""
    model, input_ids = build_model_and_input(dtype=torch.bfloat16)
    model = torch.compile(model)

    elapsed = time_loop(optimized_loop, model, input_ids, max_new_tokens,
                         n_warmup=1, n_iters=3)

    generated = optimized_loop(model, input_ids, max_new_tokens)
    return elapsed, generated

## Self-check (small + fast, runs anywhere)

In [5]:
# SELF-CHECK — DO NOT EDIT.
_n = 24  # short, for speed
_model, _ids = build_model_and_input(torch.float32)

_ref = baseline_loop(_model, _ids, _n)
_cand = optimized_loop(_model, _ids, _n)
assert tuple(_cand.shape) == (1, _n), f"expected shape (1, {_n}), got {tuple(_cand.shape)}"
assert check_correctness(_ref, _cand), \
    "optimized_loop must reproduce the baseline greedy tokens EXACTLY (fp32)"
print("optimized_loop matches baseline   PASS")

_trace = os.path.join(RESULTS_DIR, "trace_check.json")
if os.path.exists(_trace):
    os.remove(_trace)
profile(baseline_loop, _model, _ids, 4, _trace)
assert os.path.exists(_trace) and os.path.getsize(_trace) > 0, \
    "profile() must write a non-empty Chrome trace file"
print("profile() writes a Chrome trace   PASS")
print()
print("All checks passed")

optimized_loop matches baseline   PASS
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                               aten::mm         7.36%       1.026ms        10.44%       1.455ms      24.257us       1.123ms        49.55%       1.123ms      18.719us            60  
sm80_xmma_gemm_f32f32_f32f32_f32_tn_n_tilesize128x25...         0.00%       0.000us         0.00%       0.000us       0.000us     354.243us        15.63%     354.243us 

## The full run — baseline vs optimized (GPU)

Prints your speedup and writes the two traces for the writeup.

In [6]:
# DO NOT EDIT — the full run.
N_NEW = MAX_NEW_TOKENS if torch.cuda.is_available() else 32
if N_NEW != MAX_NEW_TOKENS:
    print("CPU detected — trimmed to 32 new tokens as a smoke test. "
          "REPORTED NUMBERS MUST COME FROM A GPU RUN.")

# Baseline (V0), fp32 — also the correctness reference.
model_fp32, input_ids = build_model_and_input(torch.float32)
base_t = time_loop(baseline_loop, model_fp32, input_ids, N_NEW)
ref = baseline_loop(model_fp32, input_ids, N_NEW)
print(f"baseline V0: {base_t*1e3:.1f} ms/generation")

# Correctness of your loop on the SAME fp32 model.
cand = optimized_loop(model_fp32, input_ids, N_NEW)
ok = check_correctness(ref, cand)
print(f"optimized_loop correctness (fp32): {'PASS' if ok else 'FAIL'}")

# Optimized end-to-end timing (your choice of dtype/compile/graphs).
opt_t, _ = generate_optimized(N_NEW)
speedup = base_t / opt_t
print(f"optimized:   {opt_t*1e3:.1f} ms/generation")
print(f"speedup: {speedup:.2f}x  ->  {speedup_tier(speedup)}")
if not ok:
    print("WARNING: correctness FAILED — the speedup does not count.")

# Two traces for the writeup: baseline vs optimized.
profile(baseline_loop, model_fp32, input_ids, 32,
        os.path.join(RESULTS_DIR, "trace_baseline.json"))
profile(optimized_loop, model_fp32, input_ids, 32,
        os.path.join(RESULTS_DIR, "trace_optimized.json"))
print("wrote traces to results/hw2/ — open them at https://ui.perfetto.dev")

baseline V0: 590.6 ms/generation
optimized_loop correctness (fp32): PASS


/home/labs/antebilab/guyilan/Courses/Nebius/.venv/lib/python3.9/site-packages/torch/_inductor/compile_fx.py:282: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(


optimized:   83.7 ms/generation
speedup: 7.06x  ->  >= 4.0x  (excellent)
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                               aten::mm         8.45%       8.137ms        11.91%      11.478ms      23.913us       9.491ms        50.40%       9.491ms      19.773us           480  
sm80_xmma_gemm_f32f32_f32f32_f32_tn_n_tilesize64x256...         0.00%       0.000us         0.00%       0.000us       0.000us       3.

---
## WRITEUP (be concrete and quantitative)

### Q1

From your **baseline trace**, what dominates the time? Name the specific symptom
you saw in the profiler (e.g. kernel-launch gaps, a long run of tiny kernels,
host syncs) and explain it.

**Your answer:**

The baseline trace is dominated by host syncs from the .item() call every step. Each step, the CPU stops and waits for the GPU to finish before continuing, instead of letting work queue up ahead of time. This shows up as gaps between kernel launches in the trace, so the GPU sits idle waiting on the CPU instead of running back-to-back. On top of that, each step reprocesses the whole growing sequence, so the kernels themselves get bigger and slower as generation continues.

In [7]:
# Ablation: measure each optimization's contribution one at a time.

# 1. fp32 + KV cache + no per-step sync (no bf16, no compile)
t_kv = time_loop(optimized_loop, model_fp32, input_ids, N_NEW)
print(f"+KV cache (fp32):      {t_kv*1e3:.1f} ms/generation")

# 2. + bf16 (still no compile)
model_bf16, _ = build_model_and_input(torch.bfloat16)
t_bf16 = time_loop(optimized_loop, model_bf16, input_ids, N_NEW)
print(f"+bf16:                 {t_bf16*1e3:.1f} ms/generation")

# 3. + torch.compile (this is your full generate_optimized result)
model_bf16_c, _ = build_model_and_input(torch.bfloat16)
model_bf16_c = torch.compile(model_bf16_c)
t_compiled = time_loop(optimized_loop, model_bf16_c, input_ids, N_NEW, n_warmup=1)
print(f"+torch.compile:        {t_compiled*1e3:.1f} ms/generation")

print()
print(f"baseline:              {base_t*1e3:.1f} ms/generation")
print(f"speedup +KV cache:      {base_t/t_kv:.2f}x")
print(f"speedup +bf16:          {base_t/t_bf16:.2f}x")
print(f"speedup +compile:       {base_t/t_compiled:.2f}x")

+KV cache (fp32):      168.5 ms/generation
+bf16:                 199.0 ms/generation
+torch.compile:        83.5 ms/generation

baseline:              590.6 ms/generation
speedup +KV cache:      3.50x
speedup +bf16:          2.97x
speedup +compile:       7.07x


### Q2

List each optimization you applied and the speedup it contributed — measure them
**one at a time** (baseline → +KV cache → +compile → …). A small table is ideal.

**Your answer:**

| Step | Time | Speedup vs baseline |
|---|---|---|
| Baseline | 590.6 ms | 1.0× |
| + KV cache | 168.5 ms | 3.50× |
| + bf16 | 199.0 ms | 2.97× (slower) |
| + compile | 83.5 ms | 7.07× |

KV cache gives the biggest win by skipping recomputation. bf16 alone was actually slower, since the model is small and launch overhead dominates. Compile fixes that by fusing ops, giving the best result, 7.07x overall.

### Q3

Which single change had the biggest impact, and why does it help THIS workload
(128 short decode steps on a tiny model) specifically?

**Your answer:**

The KV cache had the biggest impact. This workload is 128 tiny decode steps, so without caching, every step reprocesses the whole growing sequence from scratch, even though only one new token actually needs work. With a tiny model, that wasted recomputation is most of the cost, so skipping it gives the largest win. bf16 and compile only help with the work that's left, which is already small here.

### Q4

Your decode loop is memory-bandwidth-bound (L1 §06, L2 §01). Which of your
optimizations attack memory traffic vs CPU/launch overhead? Which kind mattered
more here, and why?

**Your answer:**

KV cache and bf16 cut memory traffic. No-sync and compile cut launch overhead.
Launch overhead mattered more here, since the model is tiny and run 128 times with many small ops. That's why compile gave a bigger jump than bf16 alone.